# Gemini Polyglot Guardian

Multilingual content-safety pipeline (Gemini + LangGraph + Gradio).
Flow: language detect/translate → safety classify → optional African cultural context → risk assess.

Full project: https://github.com/Sama-ndari/gemini-polyglot-guardian

Requires `GEMINI_API_KEY` in `.env`. Use the course `uv` environment.

In [ ]:
import json
import os
import uuid
from typing import Any, Dict, List, Literal, Optional

import gradio as gr
from dotenv import load_dotenv
from google import genai
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from typing_extensions import Annotated, TypedDict

load_dotenv(override=True)

api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("Set GEMINI_API_KEY in .env before running this notebook.")

client = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-2.0-flash"

AFRICAN_LANGUAGES = [
    "Kirundi", "Kinyarwanda", "Swahili", "Yoruba", "Hausa", "Igbo",
    "Amharic", "Zulu", "Lingala", "Wolof", "Shona", "Twi",
]

In [ ]:
class GuardianState(TypedDict):
    input_text: str
    language_hint: Optional[str]
    detected_language: str
    translation: str
    is_african_language: bool
    safety_category: str
    threat_indicators: List[str]
    safety_reasoning: str
    cultural_notes: str
    regional_context: str
    risk_level: str
    final_explanation: str
    suggested_action: str
    agent_messages: Annotated[List[str], lambda x, y: x + y]


def call_gemini(prompt: str, temperature: float = 0.3) -> str:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config={"temperature": temperature, "max_output_tokens": 1024},
    )
    return response.text or ""


def parse_json(text: str) -> Dict[str, Any]:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(cleaned)

In [ ]:
def language_detective(state: GuardianState) -> Dict[str, Any]:
    hint = state.get("language_hint") or ""
    prompt = f"""You are a language detective specializing in African languages.
Detect the language and translate to English.

TEXT:
\"\"\"{state['input_text']}\"\"\"
{f'HINT: {hint}' if hint else ''}

African focus: {', '.join(AFRICAN_LANGUAGES)}

Return ONLY JSON:
{{"detected_language": "...", "translation": "...", "is_african": true/false}}"""
    try:
        data = parse_json(call_gemini(prompt))
        lang = data.get("detected_language", "Unknown")
        return {
            "detected_language": lang,
            "translation": data.get("translation", state["input_text"]),
            "is_african_language": bool(data.get("is_african", False)),
            "agent_messages": [f"Language Detective: {lang}"],
        }
    except Exception:
        return {
            "detected_language": "Unknown",
            "translation": state["input_text"],
            "is_african_language": False,
            "agent_messages": ["Language Detective: parse failed"],
        }


def safety_analyzer(state: GuardianState) -> Dict[str, Any]:
    prompt = f"""Classify content safety.
Categories: Safe | Misinformation | Scam | Hate Speech | Manipulation

ORIGINAL ({state['detected_language']}):
\"\"\"{state['input_text']}\"\"\"

ENGLISH:
\"\"\"{state['translation']}\"\"\"

Return ONLY JSON:
{{"category": "...", "threat_indicators": ["..."], "reasoning": "..."}}"""
    try:
        data = parse_json(call_gemini(prompt))
        cat = data.get("category", "Safe")
        return {
            "safety_category": cat,
            "threat_indicators": data.get("threat_indicators", []),
            "safety_reasoning": data.get("reasoning", ""),
            "agent_messages": [f"Safety Analyzer: {cat}"],
        }
    except Exception:
        return {
            "safety_category": "Safe",
            "threat_indicators": [],
            "safety_reasoning": "parse failed",
            "agent_messages": ["Safety Analyzer: parse failed"],
        }


def cultural_context_agent(state: GuardianState) -> Dict[str, Any]:
    prompt = f"""African cultural context for safety review.
Language: {state['detected_language']}
Text: \"\"\"{state['input_text']}\"\"\"
Translation: \"\"\"{state['translation']}\"\"\"
Initial category: {state['safety_category']}

Note regional patterns (e.g. mobile-money scams) if relevant.
Return ONLY JSON:
{{"cultural_notes": "...", "regional_context": "..."}}"""
    try:
        data = parse_json(call_gemini(prompt))
        return {
            "cultural_notes": data.get("cultural_notes", ""),
            "regional_context": data.get("regional_context", ""),
            "agent_messages": [f"Cultural Context: {state['detected_language']}"],
        }
    except Exception:
        return {
            "cultural_notes": "",
            "regional_context": "",
            "agent_messages": ["Cultural Context: parse failed"],
        }


def risk_assessor(state: GuardianState) -> Dict[str, Any]:
    cultural = ""
    if state.get("is_african_language") and state.get("cultural_notes"):
        cultural = (
            f"Cultural notes: {state.get('cultural_notes')}\n"
            f"Region: {state.get('regional_context')}\n"
        )
    prompt = f"""Final risk assessment.
Language: {state['detected_language']} (African={state.get('is_african_language')})
Translation: {state['translation']}
Category: {state['safety_category']}
Indicators: {', '.join(state.get('threat_indicators') or []) or 'none'}
Reasoning: {state.get('safety_reasoning')}
{cultural}
Risk levels: None | Low | Medium | High | Critical
Return ONLY JSON:
{{"risk_level": "...", "final_explanation": "...", "suggested_action": "..."}}"""
    try:
        data = parse_json(call_gemini(prompt))
        level = data.get("risk_level", "None")
        return {
            "risk_level": level,
            "final_explanation": data.get("final_explanation", ""),
            "suggested_action": data.get("suggested_action", ""),
            "agent_messages": [f"Risk Assessor: {level}"],
        }
    except Exception:
        return {
            "risk_level": "None",
            "final_explanation": "parse failed",
            "suggested_action": "Retry",
            "agent_messages": ["Risk Assessor: parse failed"],
        }

In [ ]:
def route_after_safety(state: GuardianState) -> Literal["cultural_context", "risk_assessor"]:
    if state.get("is_african_language", False):
        return "cultural_context"
    return "risk_assessor"


def build_guardian_graph():
    builder = StateGraph(GuardianState)
    builder.add_node("language_detective", language_detective)
    builder.add_node("safety_analyzer", safety_analyzer)
    builder.add_node("cultural_context", cultural_context_agent)
    builder.add_node("risk_assessor", risk_assessor)
    builder.add_edge(START, "language_detective")
    builder.add_edge("language_detective", "safety_analyzer")
    builder.add_conditional_edges(
        "safety_analyzer",
        route_after_safety,
        {"cultural_context": "cultural_context", "risk_assessor": "risk_assessor"},
    )
    builder.add_edge("cultural_context", "risk_assessor")
    builder.add_edge("risk_assessor", END)
    return builder.compile(checkpointer=MemorySaver())


guardian_graph = build_guardian_graph()


def analyze(text: str, language_hint: Optional[str] = None) -> Dict[str, Any]:
    initial = {
        "input_text": text,
        "language_hint": language_hint,
        "detected_language": "",
        "translation": "",
        "is_african_language": False,
        "safety_category": "",
        "threat_indicators": [],
        "safety_reasoning": "",
        "cultural_notes": "",
        "regional_context": "",
        "risk_level": "",
        "final_explanation": "",
        "suggested_action": "",
        "agent_messages": [],
    }
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    return guardian_graph.invoke(initial, config=config)

In [ ]:
EXAMPLES = {
    "kirundi_safe": "Mwiriwe neza! Ndagukunda cane. Umunsi mwiza! Amahoro.",
    "swahili_scam": (
        "Umeshinda shilingi milioni 10! Tuma nambari yako ya simu na malipo "
        "ya usajili wa shilingi 5000 kupitia M-Pesa sasa hivi!"
    ),
    "english_safe": "Good morning. Looking forward to our meeting tomorrow.",
}


def analyze_for_ui(text: str, language_hint: str = "Auto-detect") -> tuple:
    if not text or not text.strip():
        return "Enter some text.", "", "{}"
    hint = None if language_hint == "Auto-detect" else language_hint
    result = analyze(text, language_hint=hint)
    log = "\n".join(f"- {m}" for m in result.get("agent_messages", []))
    summary = (
        f"**Language:** {result.get('detected_language')}\n\n"
        f"**Category:** {result.get('safety_category')} | "
        f"**Risk:** {result.get('risk_level')}\n\n"
        f"**Translation:** {result.get('translation')}\n\n"
        f"**Explanation:** {result.get('final_explanation')}\n\n"
        f"**Action:** {result.get('suggested_action')}"
    )
    payload = {
        "language": result.get("detected_language"),
        "is_african_language": result.get("is_african_language"),
        "translation": result.get("translation"),
        "category": result.get("safety_category"),
        "risk_level": result.get("risk_level"),
        "threat_indicators": result.get("threat_indicators", []),
        "cultural_notes": result.get("cultural_notes", ""),
        "suggested_action": result.get("suggested_action"),
    }
    return summary, log, json.dumps(payload, indent=2, ensure_ascii=False)


with gr.Blocks(title="Gemini Polyglot Guardian") as app:
    gr.Markdown("## Gemini Polyglot Guardian\nLangGraph safety pipeline for multilingual text.")
    with gr.Row():
        with gr.Column():
            inp = gr.Textbox(label="Text", lines=5, placeholder="Paste text in any language")
            lang = gr.Dropdown(
                label="Language hint",
                choices=["Auto-detect", "Kirundi", "Swahili", "English", "French"],
                value="Auto-detect",
            )
            btn = gr.Button("Analyze", variant="primary")
            with gr.Row():
                gr.Button("Kirundi safe").click(
                    lambda: EXAMPLES["kirundi_safe"], outputs=inp
                )
                gr.Button("Swahili scam").click(
                    lambda: EXAMPLES["swahili_scam"], outputs=inp
                )
                gr.Button("English safe").click(
                    lambda: EXAMPLES["english_safe"], outputs=inp
                )
        with gr.Column():
            out_summary = gr.Markdown()
            out_log = gr.Markdown(label="Agent log")
            out_json = gr.Code(language="json", label="JSON")
    btn.click(analyze_for_ui, inputs=[inp, lang], outputs=[out_summary, out_log, out_json])

app.launch(share=False)